# オプション課題：画像を水増しすると過学習は減るか（データ拡張）

**問い**：訓練画像を毎回少しずつ変えて（左右反転・回転・明るさ）学習すると，第1回 GW3 試行③の過学習は減るか．

演習3 の 3 つの方法は「モデルや学習のしかた」を変えた．データ拡張は **データの側** を変える 4 つ目の方法で，教科書 第5章ではコードテンプレート②を `data_transforms`（訓練用と検証用）に分けて書く．
教科書には「Cable1_Dataset では回転などを控えめにした方がよい」とある．本当か確かめる．
3〜4 グループで組み，4 つの条件 × 乱数の種を分担する．提出は任意．

1. 実行する前に (1) に予測を書く
2. (2) に自分の担当（`AUG` と `SEED`）を書き，(3) を実行する
3. (4) の手順で CSV を仲間と共有し，(5) で表を作り，(6) の考察を書く（提出）

## (0) グループと役割の設定

In [ ]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 1                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) 予測（実行する前に書く）

このセルをダブルクリックして書き込む．

- データ拡張なし（none）と，全部入り（all）の検証正解率：none ＿＿％，all ＿＿％
- 汎化ギャップ（訓練正解率 − 検証正解率）が一番小さくなる条件：＿＿

In [ ]:
# (2) 自分の担当を書く
AUG = "none"     # "none"（なし）/ "flip"（左右反転）/ "rotate"（±10 度回転）/ "all"（反転＋回転＋明るさ）
SEED = 0         # 0〜4 のどれか（同じ AUG を別の SEED で実行すると，値のぶれが見える）

In [ ]:
# (3) 学習して，検証データとテストデータの正解率を記録する（40 エポック，30 秒ほど）
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from common.cable import loaders, make_cnn
from common.device import get_device
from common.logger import ResultLogger

AUGS = {
    "none": [],
    "flip": [transforms.RandomHorizontalFlip()],           # 50% の確率で左右反転
    "rotate": [transforms.RandomRotation(10)],             # −10〜+10 度の範囲で回転
    "all": [transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), transforms.ColorJitter(brightness=0.2)],
}
# コードテンプレート②：データ前処理 — 過学習対応（データ拡張は訓練データだけにかける）
data_transforms = {
    "train": transforms.Compose([transforms.Resize((64, 64)), *AUGS[AUG], transforms.ToTensor()]),
    "val": transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()]),
}
device = get_device()
# コードテンプレート③④（GW3 と同じ分け方．SEED では変えない）．各データは CustomDataset に入っているので，変換を差し替えられる
train_loader, val_loader, test_loader, _ = loaders(seed=0)
train_loader.dataset.transform = data_transforms["train"]
plain_train = DataLoader(loaders(seed=0)[0].dataset, batch_size=8)    # 訓練正解率は拡張なしの画像（"val" の変換）で測る
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

model = make_cnn().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def accuracy(loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            correct += (model(images.to(device)).argmax(1).cpu() == labels).sum().item(); total += len(labels)
    return correct / total

for epoch in range(40):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(); criterion(model(images), labels).backward(); optimizer.step()
r = {"train_acc": accuracy(plain_train), "val_acc": accuracy(val_loader), "test_acc": accuracy(test_loader)}
r["gap"] = r["train_acc"] - r["val_acc"]
print(f"{AUG} seed={SEED}  訓練 {r['train_acc'] * 100:.1f}%  検証 {r['val_acc'] * 100:.1f}%  テスト {r['test_acc'] * 100:.1f}%  ギャップ {r['gap'] * 100:.1f}")
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d3", exercise="opt3", device=device)
logger.log_many(r, condition=AUG, seed=SEED)
print("書き込み先:", logger.path)

## (4) 共有する

1. 自分の `results/c1_d3_opt3_<グループ>.csv` を，組んだ仲間全員に送る
2. 受け取った CSV は，すべて `results/opt3_shared/` フォルダに入れる（無ければ作る．同じ名前は末尾に `_2` などを付ける）
3. もう 1 条件実行したいときは，(2) を書き換えて (3) を実行し直す

In [ ]:
# (5) 集まった全員分を表にする（何回実行してもよい）．検証・テストは 30 枚ずつなので 1 枚で 3.3 ポイント動く
import glob
import pandas as pd
files = glob.glob(str(ROOT / "results" / "c1_d3_opt3_*.csv")) + glob.glob(str(ROOT / "results" / "opt3_shared" / "*.csv"))
df = pd.concat([pd.read_csv(f) for f in files]).drop_duplicates()
wide = df.pivot_table(index=["condition", "seed", "group_id"], columns="metric_name", values="metric_value").reset_index()
g = wide.groupby("condition")
summary = (g[["train_acc", "val_acc", "test_acc", "gap"]].mean() * 100).add_suffix("_平均")
summary["val_acc_標準偏差"] = g["val_acc"].std() * 100
summary["実行回数"] = g.size()
summary.round(1)

## (6) 考察（提出）

このセルをダブルクリックして，＿＿ に書き込む（それぞれ 1〜2 文）．

1. 予測と比べて，当たったこと・外れたこと：＿＿
2. データ拡張で汎化ギャップは小さくなったか．どの拡張が一番効いたか：＿＿
3. 条件の差は，同じ条件を別の SEED で実行したときのぶれより大きいか：＿＿
4. 教科書の「Cable1_Dataset では回転などを控えめにした方がよい」は，この結果で支持されたか．ケーブル画像で拡張が逆効果になりうる理由も書く：＿＿